# 05 Model Explainability
Inspect which features contribute most to churn predictions.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")


In [ ]:
DATA_PATH = "../data/raw/Telco_customer_churn.xlsx"

df = pd.read_excel(DATA_PATH)
df.head()


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

feature_cols = [
    'Senior Citizen', 'Partner', 'Dependents', 'Internet Service',
    'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
    'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Months',
    'Monthly Charges', 'CLTV', 'Total Charges'
]

X = df[feature_cols].copy()
y = df['Churn Value']

X['Total Charges'] = pd.to_numeric(X['Total Charges'], errors='coerce')

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore')),
        ('scaler', StandardScaler(with_mean=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, numerical_cols),
        ('cat_pipeline', cat_pipeline, categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
print(f"Random Forest ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")


In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df.head(20)


In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(20), x='importance', y='feature')
plt.title('Top 20 Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc'
)

perm_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

perm_importance_df.head(20)


In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=perm_importance_df.head(20), x='importance_mean', y='feature')
plt.title('Top 20 Permutation Importances')
plt.xlabel('Mean ROC AUC Drop')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
try:
    import shap

    sample_size = min(500, X_test.shape[0])
    X_test_sample = X_test[:sample_size]

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_sample)

    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values

    shap.summary_plot(shap_values_to_plot, X_test_sample, feature_names=feature_names, show=True)
except ImportError:
    print('shap is not installed. Install it to run SHAP explainability plots.')
